# 00 · מבוא ועריכת שפיות (Intro & sanity edit)

מטרת המחברת: (1) לטעון את המודל, (2) לראות תשובות בסיסיות, (3) להריץ עריכת PISCES ידועה (מחיקת 'הארי פוטר' מהמאמר) ולוודא שהמנגנון עובד, (4) להכיר את פונקציות העזר ב-`student_utils`.

**לכל ניסוי שאלו:** למה עושים? מה עושים? מה קיבלנו?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
_d = os.getcwd()
while not os.path.exists(os.path.join(_d, 'editor.py')) and _d != os.path.dirname(_d):
    _d = os.path.dirname(_d)
os.chdir(_d); sys.path.insert(0, _d)
print('repo root:', _d)

## 1. טעינת המודל

In [ ]:
from student_utils.model_loading import load_student_model, get_default_generation_config
model, tm = load_student_model()  # google/gemma-2-2b-it on cuda
print(get_default_generation_config())

## 2. תשובות בסיס (baseline)

In [ ]:
from student_utils.generation import generate_one
for q in ['What is the capital of France?', "What are Harry Potter's parents' names?"]:
    print('Q:', q)
    print('A:', generate_one(tm, q, max_new_tokens=80))
    print('-'*80)

## 3. עריכת שפיות: מחיקת 'הארי פוטר' (מהמאמר)
אנחנו משתמשים בפיצ'רים ובהיפר-פרמטרים מהמאמר. אם זה עובד — נראה שהמודל מאבד ידע על הארי פוטר, בעוד שאלות לא קשורות נשארות תקינות. כך מוודאים שהעריכה פועלת מקצה לקצה.

In [ ]:
# Harry Potter feature set from the paper (sign=-1 means suppress; maps to Feature(neg=True))
hp_feature_set = {
    'name': 'harry_potter_demo',
    'description': 'Paper features for erasing the Harry Potter concept.',
    'features': [
        {'layer': 1,  'feature_id': 8965,  'sign': -1, 'why': 'paper'},
        {'layer': 1,  'feature_id': 13394, 'sign':  1, 'why': 'paper'},
        {'layer': 4,  'feature_id': 661,   'sign': -1, 'why': 'paper'},
        {'layer': 20, 'feature_id': 11104, 'sign': -1, 'why': 'paper'},
        {'layer': 20, 'feature_id': 14668, 'sign':  1, 'why': 'paper'},
    ],
}
edit_config = {'tau': 0.4, 'mu': 36, 'linscale': True, 'use_signs': False, 'description': 'HP demo'}

In [ ]:
from student_utils.pisces_adapter import temporary_pisces_edit
from student_utils.generation import generate_many, compare_generations_dataframe
hp_qs = ["What are Harry Potter's parents' names?",
         'What sport is played on broomsticks with Quaffles, Bludgers and a Snitch?']
control_qs = ['What is the capital of France?', "What's the distance to the moon?"]
prompts = hp_qs + control_qs
baseline = generate_many(tm, prompts, max_new_tokens=100)
with temporary_pisces_edit(model, hp_feature_set, edit_config):
    edited = generate_many(tm, prompts, max_new_tokens=100)
compare_generations_dataframe(prompts, baseline, edited)

מצופה: התשובות על הארי פוטר משתנות מהותית, התשובות הלא-קשורות כמעט זהות. שימו לב: העריכה מתבטלת אוטומטית ביציאה מה-`with` (אין הצטברות עריכות).

## 4. סיור בפונקציות העזר
כל המודולים נמצאים ב-`student_utils/`. קראו את הקוד של כל אחד והסבירו במילים שלכם מה הוא עושה.

In [ ]:
import student_utils.datasets, student_utils.scoring, student_utils.pisces_adapter
import student_utils.feature_search, student_utils.generation, student_utils.reporting
for m in [student_utils.datasets, student_utils.scoring, student_utils.pisces_adapter,
          student_utils.feature_search, student_utils.generation, student_utils.reporting]:
    print(m.__name__, '->', [x for x in dir(m) if not x.startswith('_')][:12])

## 5. הצעד הבא
עברו למחברת `01` של הנושא שלכם (gaia/ או itay/).